In [ ]:
# Data source: https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis?utm_source=chatgpt.com

In [5]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [6]:
data = pd.read_csv("marketing_campaign.csv", sep=None, engine='python')
data.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


In [7]:
len(data)

2240

In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   i

In [4]:
import pandas as pd
import numpy as np
import random

# Load the dataset
# (Assumes you've downloaded the Kaggle "Customer Personality Analysis" CSV to local)
df = pd.read_csv("marketing_campaign.csv", sep=None, engine='python')

# Preprocess / define simple state features (discretized)
# Let's pick Recency, Income, and Total Spend = sum of Mnt* columns
df['Total_Spend'] = (df['MntWines'] + df['MntFruits'] + 
                     df['MntMeatProducts'] + df['MntSweetProducts'] + df['MntFishProducts'] + df['MntGoldProds'])

# Discretize into bins
df['recency_bin'] = pd.qcut(df['Recency'], 5, labels=False)
df['income_bin'] = pd.qcut(df['Income'].fillna(df['Income'].median()), 5, labels=False)
df['spend_bin'] = pd.qcut(df['Total_Spend'], 5, labels=False)

# Define state as a tuple
df['state'] = list(zip(df['recency_bin'], df['income_bin'], df['spend_bin']))

# Possible actions: for simplicity, 3 actions: 0 = no offer, 1 = small coupon, 2 = big coupon
actions = [0, 1, 2]

# Initialize Q-table: state → action values
# because state space is (5 x 5 x 5) = 125 possible states
num_states = 5 * 5 * 5
num_actions = len(actions)
Q = np.zeros((num_states, num_actions))

# Map state tuple to index
state_to_idx = {state: idx for idx, state in enumerate(df['state'].unique())}

# Hyperparameters
alpha = 0.1   # learning rate
gamma = 0.9   # discount factor
epsilon = 0.1  # exploration rate

# Simulate episodes (this is toy: we sample customers randomly, take an action, simulate reward)
def simulate_reward(action, customer):
    """
    Simulate a simple reward:
    - If action = 2 (big coupon), high spenders may buy more => positive reward
    - If action = 0, maybe only high income or frequent customers buy.
    This is just a toy model.
    """
    base = customer['Total_Spend'] / 1000.0  # scale
    if action == 0:
        return base * 0.5  # no offer, lower conversion
    elif action == 1:
        return base * 1.0  # small coupon
    elif action == 2:
        # big coupon: attracts low spenders, but cost is higher
        return base * 1.2 - 0.5  # subtract cost
    return 0

# Training loop
for episode in range(10000):
    # sample a random customer
    customer = df.sample(1).iloc[0]
    s_idx = state_to_idx[customer['state']]
    
    # choose action (epsilon-greedy)
    if random.random() < epsilon:
        action = random.choice(actions)
    else:
        action = actions[np.argmax(Q[s_idx])]
    
    # get reward
    r = simulate_reward(action, customer)
    
    # pick a "next state": for toy, assume state doesn't change or re-sample
    next_customer = df.sample(1).iloc[0]
    ns_idx = state_to_idx[next_customer['state']]
    
    # Q-learning update
    Q[s_idx, actions.index(action)] = Q[s_idx, actions.index(action)] + \
        alpha * (r + gamma * np.max(Q[ns_idx]) - Q[s_idx, actions.index(action)])

# After training, inspect policy
policy = {state: actions[np.argmax(Q[idx])] for state, idx in state_to_idx.items()}

print("Learned policy (state → best action):")
for state, act in policy.items():
    print(state, "→", act)


Learned policy (state → best action):
(2, 2, 4) → 0
(1, 2, 0) → 0
(1, 3, 3) → 0
(1, 0, 0) → 0
(4, 3, 2) → 0
(0, 3, 3) → 1
(1, 2, 2) → 0
(1, 1, 1) → 0
(0, 0, 0) → 0
(3, 0, 0) → 0
(0, 2, 0) → 0
(2, 0, 1) → 0
(4, 3, 3) → 0
(2, 3, 2) → 0
(1, 4, 4) → 0
(2, 1, 1) → 0
(1, 1, 2) → 0
(4, 4, 4) → 0
(4, 1, 1) → 0
(2, 1, 2) → 0
(2, 0, 4) → 1
(3, 3, 3) → 0
(0, 3, 2) → 0
(3, 1, 2) → 0
(4, 0, 1) → 0
(0, 2, 2) → 0
(0, 2, 3) → 2
(0, 3, 4) → 2
(2, 2, 1) → 0
(3, 0, 1) → 0
(2, 2, 3) → 0
(0, 4, 4) → 1
(4, 0, 0) → 0
(4, 2, 1) → 0
(3, 4, 3) → 1
(1, 0, 1) → 0
(4, 1, 0) → 0
(1, 3, 2) → 0
(2, 4, 4) → 0
(2, 2, 0) → 0
(3, 2, 2) → 0
(0, 1, 0) → 1
(1, 2, 1) → 0
(3, 1, 0) → 0
(0, 4, 3) → 1
(3, 4, 4) → 0
(3, 2, 1) → 0
(4, 2, 3) → 0
(2, 4, 3) → 0
(0, 2, 1) → 0
(2, 1, 0) → 0
(1, 4, 3) → 1
(0, 1, 1) → 0
(2, 3, 4) → 0
(4, 4, 3) → 0
(0, 1, 2) → 0
(3, 2, 3) → 0
(1, 3, 4) → 0
(3, 1, 1) → 0
(1, 1, 0) → 0
(2, 3, 3) → 0
(4, 1, 2) → 0
(2, 3, 0) → 0
(0, 0, 1) → 1
(2, 0, 0) → 1
(4, 3, 4) → 0
(4, 2, 2) → 1
(2, 2, 2) → 0
(3, 3, 4) 